In [241]:
import os
import json
import re

import pandas as pd

In [242]:
name_pattern = r'\n# [A-Za-z ]+\n'

In [ ]:
# runs = [
#     os.path.join('experiments', str(x)) for x in range(744, 824)
# ] # without +/- fix in Z_0 calculation

runs = [
    os.path.join('experiments', str(x)) for x in range(824, 904)
] # fixed +/- in Z_0 calculation

assert len(runs) == 80

In [244]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf vibration',
    4: 'Random solution'
}

In [245]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error',
        'time_to_1p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item()
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] == min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item()
    
    # epoch where we first reach 5% relative error
    first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item()

    first_1p_epoch = int(log[log['rel_l2_error'] < 0.01].iloc[0]['epoch'])
    info['time_to_1p_rel_l2_error'] = log[log['epoch'] <= first_1p_epoch]['training_time'].sum().item()
    
    df.loc[len(df)] = info

In [246]:
results = df[[
    'soln_name',
    'time_to_5p_rel_l2_error',
    'time_to_1p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'use_cpu',
    'add_bc',
    'seed'
]].rename(columns={
    'time_to_1p_rel_l2_error': 'Time to <1% err. (s)',
    'time_to_5p_rel_l2_error': 'Time to <5% err. (s)',
    'time_to_min_rel_l2_error': 'Time to min. L2 err. (s)',
    'min_rel_l2_error': 'Min. L2 err. (%)',
    'soln_name': 'Solution',
    'use_cpu': 'on CPU',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'on CPU', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        display_results[col] = mean.map('{:.2e}'.format) + ' ± ' + sem.map('{:.2e}'.format)

display_results = pd.DataFrame(display_results)
display_results

Time to <5% err. (s) Time to <1% err. (s)  \
Solution        on CPU w/ BC                                             
Hopf vibration  False  False            0.8 ± 0.0            2.5 ± 0.0   
                       True             2.2 ± 0.6            6.6 ± 3.3   
                True   False            0.7 ± 0.1            2.9 ± 0.2   
                       True             1.8 ± 0.1            3.9 ± 0.2   
Plane waves     False  False            3.3 ± 0.1            5.9 ± 0.2   
                       True             3.5 ± 0.5            9.9 ± 0.6   
                True   False            3.0 ± 0.1            5.4 ± 0.2   
                       True             4.5 ± 1.7           11.0 ± 1.1   
Radial waves    False  False            3.0 ± 0.1            8.4 ± 0.5   
                       True             4.2 ± 0.4           21.3 ± 3.6   
                True   False            2.6 ± 0.2            7.3 ± 0.7   
                       True             4.4 ± 0.2           23.1 ± 3.0   
Random solution False  False            1.7 ± 0.1            3.0 ± 0.2   
                       True             5.6 ± 1.1           38.6 ± 8.9   
                True   False            1.6 ± 0.1            2.9 ± 0.2   
                       True             5.1 ± 0.2           37.1 ± 1.8   

                             Time to min. L2 err. (s)     Min. L2 err. (%)  
Solution        on CPU w/ BC                                                
Hopf vibration  False  False              120.7 ± 1.2  3.10e-05 ± 2.30e-06  
                       True               123.1 ± 4.2  7.72e-04 ± 4.14e-04  
                True   False             173.8 ± 13.8  3.05e-05 ± 2.70e-06  
                       True               173.3 ± 9.8  3.35e-04 ± 2.30e-05  
Plane waves     False  False              135.4 ± 2.4  1.09e-04 ± 6.91e-06  
                       True               136.2 ± 1.2  1.29e-04 ± 2.64e-05  
                True   False              128.3 ± 3.3  1.13e-04 ± 3.20e-06  
                       True              145.0 ± 11.6  1.34e-04 ± 2.51e-05  
Radial waves    False  False              135.5 ± 1.6  4.05e-04 ± 4.12e-05  
                       True               134.8 ± 1.7  3.03e-03 ± 9.51e-04  
                True   False              128.9 ± 8.8  4.16e-04 ± 4.37e-05  
                       True               136.5 ± 3.6  3.46e-03 ± 6.69e-04  
Random solution False  False              126.7 ± 4.5  1.42e-04 ± 5.04e-06  
                       True               129.1 ± 3.4  2.51e-03 ± 5.84e-04  
                True   False             126.5 ± 11.8  1.42e-04 ± 4.52e-06  
                       True               109.1 ± 4.0  2.92e-03 ± 2.25e-04